In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

from dataSet import SGNS_store_DataSet

from copy import deepcopy

import nltk
from nltk.tokenize import word_tokenize
# nltk.download('punkt_tab') # A faire la première fois

import seaborn as sns
import matplotlib.pyplot as plt

import unicodedata
import string

from visuEmbedding import components_to_fig_3D, components_to_fig_3D_animation
from modelSGNS import OnlyOneEmb, SGNS_OneEmbWeighted, SkipGramModel, SGNS_Weighted
from data.pipData import pipe_data, prepare_data, prepare_data_with_intonation, separate_text_intonation
import tool

import numpy as np
import pandas as pd

from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import skew

from collections import Counter

from data.pipData import separate_text_intonation
from dataSet import W2V_weighted_DataSet, W2V_weighted_DataSet_v2, dataset_weighted, SGNS_store_DataSet

import random

from typing import Callable, List, Type

[nltk_data] Downloading package punkt_tab to /home/pe/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /home/pe/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


# Fct

In [9]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed) # if you use multi-GPU
    # For absolute reproducibility (may slow down training slightly):
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    
def createEmbeddingWithSeed(seed:list[int],
                            dataset:Dataset,
                            model:nn.Module,
                            re_init_fct:Callable,
                            get_embedding:Callable,
                            optimizer_cls:type[torch.optim.Optimizer],
                            lr:float, nb_epoch:int,
                            path_to_save:str,
                            words:list[str],
                            device: str = "cpu",
                            batch_size: int = 16,
                            num_workers: int = 0
                        ):
    model.to(device)
    loss_by_seed: List[List[float]] = []
    
    for s in seed:
        set_all_seeds(s)
        loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
        
        model = re_init_fct(model)
        optimizer = optimizer_cls(model.parameters(), lr=lr)
        
        loss_by_epoch:list[float] = []
        for epoch in range(nb_epoch):
            loss_in_epoch:list[float] = []
            for sentence_nb, data in enumerate(loader):
                if isinstance(data, (list, tuple)):
                    data = [d.to(device) for d in data]
                else:
                    data = data.to(device)
                    
                optimizer.zero_grad()
                loss:torch.Tensor = model(data)
                loss.backward()
                optimizer.step()
                loss_in_epoch.append(loss.detach().cpu().item())
            loss_by_epoch.append(np.mean(loss_in_epoch))
        
        vectors = get_embedding(model)
        model_name = type(model).__name__
        np.savez(
            f'{path_to_save}/seed_{s}_{model_name}_.npz', 
            vectors=vectors,
            words=words
        )
        loss_by_seed.append(loss_by_epoch)
        print(f"For the seed {s} we have loss {loss_by_epoch}")
        
    
    return loss_by_seed

# Data

## GNG

In [ ]:
data = prepare_data_with_intonation(
    file_path="./data/GoodNightGorilla_Intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={# specific to corpus 
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",},
    stop_words=["s", "n't"],
    break_line=False
)

texts, intonations = separate_text_intonation(data)

print(texts)

## I went Walking

In [11]:
data = prepare_data_with_intonation(
    file_path="data/IwentWalking_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)


## GNG + I went Walking

In [12]:
data = prepare_data_with_intonation(
    file_path="./data/GoodNightGorilla_Intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={# specific to corpus 
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",},
    stop_words=["s", "n't"],
    break_line=False
)

texts, intonations = separate_text_intonation(data)

print(texts)

data = prepare_data_with_intonation(
    file_path="data/IwentWalking_intonation.txt",
    language='english',
    remove_accent=True,
    remove_punct=True,
    keep_apostrophes=False,
    contraction_map={
        "that's" : "thatis",
        "it's" : "itis",
        "don't": "donot",
        "doesn't": "doesnot",
        "you're": "youre"
    },
    stop_words=["s", "n't"],
    break_line=False
)

texts2, intonations2 = separate_text_intonation(data)
intonations.extend(intonations2)
texts.extend(texts2)
print(texts)

[['look', 'there', 'is', 'the', 'zookeeper', 'he', 'has', 'a', 'big', 'flashlight', 'to', 'see', 'in', 'the', 'dark', 'click', 'what', 'is', 'he', 'saying', 'to', 'the', 'animal', 'he', 'says', 'good', 'night', 'gorilla', 'can', 'you', 'say', 'good', 'night'], ['oh', 'my', 'goodness', 'look', 'closer', 'is', 'the', 'gorilla', 'going', 'to', 'sleep', 'no', 'he', 'is', 'reaching', 'out', 'and', 'taking', 'the', 'keys', 'that', 'sneaky', 'gorilla', 'is', 'stealing', 'the', 'keys', 'right', 'off', 'the', 'zookeeper', 'belt', 'jingle', 'jangle'], ['who', 'sees', 'him', 'doing', 'it', 'it', 'the', 'little', 'mouse', 'squeak', 'squeak', 'the', 'mouse', 'is', 'watching', 'everything'], ['look', 'at', 'the', 'gorilla', 'room', 'he', 'has', 'a', 'bicycle', 'in', 'there', 'and', 'a', 'big', 'tire', 'to', 'swing', 'on', 'but', 'he', 'doesnot', 'want', 'to', 'stay', 'inside', 'does', 'he', 'he', 'wants', 'to', 'follow', 'the', 'zookeeper'], ['and', 'what', 'is', 'that', 'pink', 'thing', 'floating',

# SGNS_OneEmbWeighted
norm01 : range_norm = 1.9 and center_norm = 1.

norm02 : range_norm = 1.75 and center_norm = 1.

norm03 : range_norm = 1.5 and center_norm = 1.


# My method

In [15]:
range_norm = 1.75
center_norm = 1.
intonations_normalize = tool.normalize_range_center(intonations, range_normalize=range_norm, center=center_norm)

dataset:dataset_weighted = dataset_weighted(sentences=texts,
                                intonations=intonations_normalize, nb_neg=10, window_size=6)

model = SGNS_OneEmbWeighted(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

def re_init_model_w2v_weighted(model:SGNS_OneEmbWeighted):
    with torch.no_grad():
        model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
    return model

def get_embedding_w2v_weighted(model:SGNS_OneEmbWeighted):
    return model.word_emb.weight.detach().cpu().numpy()

createEmbeddingWithSeed(seed = [0, 1, 2, 3, 4],
                        dataset=dataset,
                        model=model,
                        re_init_fct=re_init_model_w2v_weighted,
                        get_embedding=get_embedding_w2v_weighted,
                        optimizer_cls=torch.optim.Adam,
                        lr=0.003, nb_epoch=50,
                        path_to_save="embedding/GNG_IWW/norm02",
                        words=list(dataset.encoder.keys()),
                        device="cuda",
                        batch_size= 16,
                        num_workers= 0)

For the seed 0 we have loss [np.float64(5.351264089561111), np.float64(5.323803057416682), np.float64(5.316924466875916), np.float64(5.310718375853253), np.float64(5.307040417667761), np.float64(5.307633416801153), np.float64(5.302556425010255), np.float64(5.309704846262071), np.float64(5.3003335305714865), np.float64(5.304499998204545), np.float64(5.302819442824336), np.float64(5.301445910969366), np.float64(5.3022085718729866), np.float64(5.304080852747824), np.float64(5.300684070328943), np.float64(5.3015057205185565), np.float64(5.299724397730311), np.float64(5.306087642639122), np.float64(5.298095421670577), np.float64(5.299812150614787), np.float64(5.297415884231833), np.float64(5.29790550099168), np.float64(5.300478454316136), np.float64(5.299182420818384), np.float64(5.299493134505912), np.float64(5.3020283237691395), np.float64(5.305555992877441), np.float64(5.297744094841317), np.float64(5.3045112356166975), np.float64(5.295864093120778), np.float64(5.298515234720836), np.flo

[[np.float64(5.351264089561111),
  np.float64(5.323803057416682),
  np.float64(5.316924466875916),
  np.float64(5.310718375853253),
  np.float64(5.307040417667761),
  np.float64(5.307633416801153),
  np.float64(5.302556425010255),
  np.float64(5.309704846262071),
  np.float64(5.3003335305714865),
  np.float64(5.304499998204545),
  np.float64(5.302819442824336),
  np.float64(5.301445910969366),
  np.float64(5.3022085718729866),
  np.float64(5.304080852747824),
  np.float64(5.300684070328943),
  np.float64(5.3015057205185565),
  np.float64(5.299724397730311),
  np.float64(5.306087642639122),
  np.float64(5.298095421670577),
  np.float64(5.299812150614787),
  np.float64(5.297415884231833),
  np.float64(5.29790550099168),
  np.float64(5.300478454316136),
  np.float64(5.299182420818384),
  np.float64(5.299493134505912),
  np.float64(5.3020283237691395),
  np.float64(5.305555992877441),
  np.float64(5.297744094841317),
  np.float64(5.3045112356166975),
  np.float64(5.295864093120778),
  np.f

# Two embedding weighted

In [ ]:
range_norm = 1.75
center_norm = 1.
intonations_normalize = tool.normalize_range_center(intonations, range_normalize=range_norm, center=center_norm)

dataset:dataset_weighted = dataset_weighted(sentences=texts,
                                intonations=intonations_normalize, nb_neg=10, window_size=6)

model = SGNS_Weighted(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

def re_init_model_w2v_weighted(model:SGNS_Weighted):
    with torch.no_grad():
        model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
        model.con_emb.weight.data.uniform_(-model.init_range, model.init_range)
    return model

def get_embedding_w2v_weighted(model:SGNS_Weighted):
    return model.word_emb.weight.detach().cpu().numpy()

createEmbeddingWithSeed(seed = [0, 1, 2, 3, 4, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19],
                        dataset=dataset,
                        model=model,
                        re_init_fct=re_init_model_w2v_weighted,
                        get_embedding=get_embedding_w2v_weighted,
                        optimizer_cls=torch.optim.Adam,
                        lr=0.003, nb_epoch=50,
                        path_to_save="embedding/norm02",
                        words=list(dataset.encoder.keys()),
                        device="cuda",
                        batch_size= 16,
                        num_workers= 0)

For the seed 0 we have loss [np.float64(3.134006625206969), np.float64(2.2703083849982826), np.float64(2.227764940786725), np.float64(2.179485166305006), np.float64(2.098391595631712), np.float64(2.0108892845966166), np.float64(1.933972659704546), np.float64(1.877777790639685), np.float64(1.8326469211634895), np.float64(1.804443158808651), np.float64(1.7876577552448987), np.float64(1.7656756444163084), np.float64(1.7546622281252178), np.float64(1.7458947109126317), np.float64(1.736423220147933), np.float64(1.732292402053061), np.float64(1.7198188177212934), np.float64(1.7202270749663016), np.float64(1.7116041440866843), np.float64(1.7107935064329525), np.float64(1.713760933514675), np.float64(1.7041595771997082), np.float64(1.7039489278240187), np.float64(1.705727851360557), np.float64(1.7011692921152366), np.float64(1.705282505390303), np.float64(1.7016348556680865), np.float64(1.6972918751177195), np.float64(1.6927077685444158), np.float64(1.6965225604910854), np.float64(1.6982495162

[[np.float64(3.134006625206969),
  np.float64(2.2703083849982826),
  np.float64(2.227764940786725),
  np.float64(2.179485166305006),
  np.float64(2.098391595631712),
  np.float64(2.0108892845966166),
  np.float64(1.933972659704546),
  np.float64(1.877777790639685),
  np.float64(1.8326469211634895),
  np.float64(1.804443158808651),
  np.float64(1.7876577552448987),
  np.float64(1.7656756444163084),
  np.float64(1.7546622281252178),
  np.float64(1.7458947109126317),
  np.float64(1.736423220147933),
  np.float64(1.732292402053061),
  np.float64(1.7198188177212934),
  np.float64(1.7202270749663016),
  np.float64(1.7116041440866843),
  np.float64(1.7107935064329525),
  np.float64(1.713760933514675),
  np.float64(1.7041595771997082),
  np.float64(1.7039489278240187),
  np.float64(1.705727851360557),
  np.float64(1.7011692921152366),
  np.float64(1.705282505390303),
  np.float64(1.7016348556680865),
  np.float64(1.6972918751177195),
  np.float64(1.6927077685444158),
  np.float64(1.69652256049

# OnlyOneEmb

In [ ]:
dataset = SGNS_store_DataSet(sentences=texts, nb_neg=10, power=0.75, subsample_thresh=0, window_size=6)

model = OnlyOneEmb(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

def re_init_model_w2v_one_emb(model:OnlyOneEmb):
    with torch.no_grad():
        model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
    return model

def get_embedding_w2v_one_emb(model:OnlyOneEmb):
    return model.word_emb.weight.detach().cpu().numpy()

createEmbeddingWithSeed(seed = [5, 6, 7, 8, 9],
                        dataset=dataset,
                        model=model,
                        re_init_fct=re_init_model_w2v_one_emb,
                        get_embedding=get_embedding_w2v_one_emb,
                        optimizer_cls=torch.optim.Adam,
                        lr=0.003, nb_epoch=50,
                        path_to_save="embedding/noNorm",
                        words=list(dataset.encoder.keys()),
                        device="cuda",
                        batch_size= 16,
                        num_workers= 0)

# SkipGramModel

In [ ]:
dataset = SGNS_store_DataSet(sentences=texts, nb_neg=10, power=0.75, subsample_thresh=0, window_size=6)

model = SkipGramModel(emb_size=dataset.vocab_size,
                            embedding_dimension=10, init_range=None, device="cuda", sparse=False)

def re_init_model_w2v_two_emb(model:SkipGramModel):
    with torch.no_grad():
        model.word_emb.weight.data.uniform_(-model.init_range, model.init_range)
        model.con_emb.weight.data.uniform_(-model.init_range, model.init_range)
    return model

def get_embedding_w2v_two_emb(model:SkipGramModel):
    return model.word_emb.weight.detach().cpu().numpy()

createEmbeddingWithSeed(seed = [5, 6, 7, 8, 9],
                        dataset=dataset,
                        model=model,
                        re_init_fct=re_init_model_w2v_two_emb,
                        get_embedding=get_embedding_w2v_two_emb,
                        optimizer_cls=torch.optim.Adam,
                        lr=0.003, nb_epoch=50,
                        path_to_save="embedding/noNorm",
                        words=list(dataset.encoder.keys()),
                        device="cuda",
                        batch_size= 16,
                        num_workers= 0)